# Continuous Controls Monitoring: Procure to Pay
Six control tests on a synthetic ledger of vendor invoices (FY 2025-26).
Input: demo_vendor_ledger_FY26.csv  |  Outputs: CSVs for the Power BI dashboard.

In [ ]:
import re
import numpy as np
import pandas as pd

FILE = "demo_vendor_ledger_FY26.csv"

# Test parameters (change these to see the exception count move)
DUP_WINDOW_DAYS = 30        # T1: same vendor and amount within this many days
APPROVAL_LIMIT  = 500000    # T2: invoices at or above this need a second approver
NEAR_BAND       = 0.10      # T2: flag amounts within 10% below the limit
SPLIT_MIN_SHARE = 0.25      # T2: each split invoice must be at least 25% of the limit
BENFORD_MIN_N   = 30        # T3: test vendors with at least 30 invoices
BENFORD_CRIT    = 26.12     # T3: chi-square critical value, 8 degrees of freedom, p = 0.001
PO_TOLERANCE    = 0.05      # T4: invoice may exceed its PO by at most 5%
NON_PO_LIMIT    = 100000    # T4: invoices without a PO must be below this
AFTER_HOURS     = (22, 6)   # T6: posted at or after 22:00 or before 06:00

## 1. Load and check the data

In [ ]:
df = pd.read_csv(FILE, dtype={"po_number": str, "invoice_no": str})
df["po_number"] = df["po_number"].fillna("")
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["posted_at"] = pd.to_datetime(df["posted_at"])

print("Rows:", len(df))
print("Total value: Rs", round(df["amount"].sum() / 1e7, 1), "Cr")
print("Vendors:", df["vendor"].nunique())
print("Duplicate invoice IDs:", df["invoice_id"].duplicated().sum())
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])

flags = []   # one row per invoice per failed test: invoice_id, test, reason
def flag(ids, test, reasons):
    for i, r in zip(ids, reasons):
        flags.append((i, test, r))

## 2. T1 Duplicate payments
Same vendor and same amount within the window, or an invoice number reused after
removing symbols, the "2526" year tag, a trailing "A" and leading zeros.

In [ ]:
t1 = df.sort_values(["vendor", "amount", "invoice_date"], kind="mergesort")
gap = t1.groupby(["vendor", "amount"])["invoice_date"].diff().dt.days
prev_id = t1.groupby(["vendor", "amount"])["invoice_id"].shift()
hit = (gap <= DUP_WINDOW_DAYS) & (t1["amount"] >= 1000)
flag(t1.loc[hit, "invoice_id"], "Duplicate payment",
     ["Same vendor and amount as " + p + ", " + str(int(g)) + " days apart" for p, g in zip(prev_id[hit], gap[hit])])

def norm_no(s):
    s = re.sub(r"[^A-Z0-9]", "", str(s).upper())
    s = s.replace("2526", "", 1)
    s = re.sub(r"A$", "", s)
    return s.lstrip("0")

df["inv_norm"] = df["invoice_no"].map(norm_no)
t1b = df.sort_values("posted_at", kind="mergesort")
rep = t1b.duplicated(["vendor", "inv_norm"], keep="first")
flag(t1b.loc[rep, "invoice_id"], "Duplicate payment",
     ["Invoice number " + n + " already used" for n in t1b.loc[rep, "invoice_no"]])

## 3. T2 Approval threshold avoidance
(a) amounts just under the approval limit; (b) same vendor, same day invoices,
each under the limit and at least 25% of it, that together reach the limit.

In [ ]:
near = (df["amount"] >= APPROVAL_LIMIT * (1 - NEAR_BAND)) & (df["amount"] < APPROVAL_LIMIT)
flag(df.loc[near, "invoice_id"], "Threshold avoidance",
     ["Rs " + format(a, ",") + " is just under the approval limit" for a in df.loc[near, "amount"]])

g = df.groupby(["vendor", "invoice_date"])["amount"]
split = ((g.transform("count") >= 2) & (g.transform("sum") >= APPROVAL_LIMIT)
         & (g.transform("max") < APPROVAL_LIMIT) & (g.transform("min") >= APPROVAL_LIMIT * SPLIT_MIN_SHARE))
flag(df.loc[split, "invoice_id"], "Threshold avoidance",
     ["Same day split invoice, group total above the limit"] * split.sum())

## 4. T3 Benford's Law, by vendor
Genuine amounts follow Benford's first digit distribution: P(d) = log10(1 + 1/d).
Chi-square test per vendor; for failing vendors, flag invoices whose leading digit
appears more than 1.25 times as often as expected.

In [ ]:
df["first_digit"] = df["amount"].astype(str).str.lstrip("0.").str[0].astype(int)
benford = np.log10(1 + 1 / np.arange(1, 10))
benford_rows = []
for vendor, grp in df.groupby("vendor"):
    n = len(grp)
    counts = grp["first_digit"].value_counts().reindex(range(1, 10), fill_value=0).values
    expected = n * benford
    chi = (((counts - expected) ** 2) / expected).sum()
    for d in range(9):
        benford_rows.append((vendor, d + 1, counts[d] / n, benford[d], n, chi))
    if n >= BENFORD_MIN_N and chi > BENFORD_CRIT:
        over = [d + 1 for d in range(9) if counts[d] > expected[d] * 1.25]
        hit = grp[grp["first_digit"].isin(over)]
        flag(hit["invoice_id"], "Benford's Law",
             ["Vendor fails Benford (chi-square " + str(round(chi, 1)) + "), leading digit " + str(d) + " over represented"
              for d in hit["first_digit"]])
        print(f"{vendor}: n={n}, chi-square={chi:.1f}, over represented digits {over}")

## 5. T4 Three way match
Invoice above its PO beyond tolerance, or no PO on an invoice at or above the non PO limit.

In [ ]:
over_po = (df["po_amount"] > 0) & (df["amount"] > df["po_amount"] * (1 + PO_TOLERANCE))
flag(df.loc[over_po, "invoice_id"], "Three way match",
     ["Exceeds PO by " + str(round((a / p - 1) * 100, 1)) + "%" for a, p in zip(df.loc[over_po, "amount"], df.loc[over_po, "po_amount"])])
no_po = (df["po_amount"] == 0) & (df["amount"] >= NON_PO_LIMIT)
flag(df.loc[no_po, "invoice_id"], "Three way match", ["No purchase order above the non PO limit"] * no_po.sum())

## 6. T5 Segregation of duties and T6 After hours posting

In [ ]:
sod = df["created_by"].notna() & (df["created_by"] == df["approved_by"])
flag(df.loc[sod, "invoice_id"], "Segregation of duties", ["Created and approved by " + u for u in df.loc[sod, "created_by"]])

hour = df["posted_at"].dt.hour
after = (df["posted_at"].dt.dayofweek == 6) | (hour >= AFTER_HOURS[0]) | (hour < AFTER_HOURS[1])
flag(df.loc[after, "invoice_id"], "After hours posting",
     ["Posted " + t.strftime("%a %H:%M") for t in df.loc[after, "posted_at"]])

## 7. Results and exports for Power BI

In [ ]:
exc = pd.DataFrame(flags, columns=["invoice_id", "test", "reason"]).drop_duplicates(["invoice_id", "test"])
exc = exc.merge(df[["invoice_id", "vendor", "amount", "posted_at"]], on="invoice_id")

ids = set(exc["invoice_id"])
df["is_exception"] = df["invoice_id"].isin(ids)
df["tests_failed"] = df["invoice_id"].map(exc.groupby("invoice_id")["test"].apply(" | ".join)).fillna("")
df["posting_month"] = df["posted_at"].dt.to_period("M").dt.to_timestamp()

summary = (exc.groupby("test").agg(exceptions=("invoice_id", "count"), value=("amount", "sum"))
              .sort_values("value", ascending=False))
summary["value_cr"] = (summary["value"] / 1e7).round(2)
print(summary[["exceptions", "value_cr"]])
print("\nUnique exceptions:", len(ids), "invoices")
print("Value flagged: Rs", round(df.loc[df["is_exception"], "amount"].sum() / 1e7, 1), "Cr of Rs",
      round(df["amount"].sum() / 1e7, 1), "Cr tested")

df.drop(columns=["inv_norm"]).to_csv("ledger_flagged.csv", index=False)
exc.to_csv("exceptions.csv", index=False)
pd.DataFrame(benford_rows, columns=["vendor", "digit", "actual_share", "benford_share", "vendor_invoices", "chi_square"]).to_csv("benford_by_vendor.csv", index=False)
print("Saved ledger_flagged.csv, exceptions.csv, benford_by_vendor.csv")